# LAB 1 — Query Rewriting: Reduzindo o Vocabulary Gap
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Implementar e comparar 3 técnicas de query rewriting em queries jurídicas reais,
medindo o impacto no retrieval antes e depois da reformulação.

**Entregáveis:**
- Pipeline de query rewriting funcional com 3 técnicas
- Tabela comparativa: query original vs reformulada para 5 queries do baseline
- Análise: qual técnica gerou maior diversidade semântica?

**Tempo estimado:** 40 minutos  
**Pré-requisito:** Aula 2 concluída — OpenSearch com corpus indexado


## ⚙️ Passo 1 — Instalação e Imports

In [ ]:
# Instalar dependências
%pip install -q langchain langchain-community httpx langchain-openai langchain-ollama langchain-groq groq openai python-dotenv sentence-transformers opensearch-py pandas numpy
print("✅ Instalação concluída")

In [ ]:
import os
import json
import time
import httpx
import pandas as pd
import numpy as np
from typing import List, Dict, Optional
from dataclasses import dataclass
from dotenv import load_dotenv

# Carrega ~/mba-rag/.env (ou .env local) se existir
load_dotenv()

# ── Configurações Groq (primário) ─────────────────────────────────
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
GROQ_LLM_MODEL = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

# ── Configurações Ollama (fallback local) ─────────────────────────
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "lama-3.1-8b")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "bge-m3")

# ── Configurações OpenSearch ──────────────────────────────────────
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
INDEX_NAME      = os.getenv("INDEX_NAME", "corpus_juridico")

print("✅ Configurações carregadas:")
print(f"   LLM primário:   Groq → {GROQ_LLM_MODEL}  ({'API key OK' if GROQ_API_KEY else 'SEM key → cairá no Ollama'})")
print(f"   LLM fallback:   Ollama → {OLLAMA_LLM_MODEL}  ({OLLAMA_BASE_URL})")
print(f"   Embeddings:     Ollama → {OLLAMA_EMBED_MODEL} (fallback: HuggingFace BAAI/bge-m3)")
print(f"   OpenSearch:     {OPENSEARCH_HOST}:{OPENSEARCH_PORT} | índice: {INDEX_NAME}")

## 🔌 Passo 2 — Conectar ao LLM e OpenSearch

In [ ]:
from openai import OpenAI

# ─────────────────────────────────────────────────────────────────
# Estratégia: Groq primário (llama-3.1-8b-instant) + Ollama fallback
# Ambos expõem API OpenAI-compatible, então usamos um único cliente.
# ─────────────────────────────────────────────────────────────────

llm_client = None
MODEL_NAME = None
LLM_PROVIDER = None

# Tentativa 1 — Groq (nuvem, baixa latência)
if GROQ_API_KEY:
    try:
        candidate = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY,http_client=httpx.Client(verify=False))
        _ = candidate.chat.completions.create(
            model=GROQ_LLM_MODEL,
            messages=[{"role": "user", "content": "ok"}],
            max_tokens=2, temperature=0.0,

        )
        llm_client, MODEL_NAME, LLM_PROVIDER = candidate, GROQ_LLM_MODEL, "groq"
        print(f"✅ LLM via Groq | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"⚠️  Groq falhou ({e}); caindo para Ollama local…")

# Tentativa 2 — Ollama (fallback local, API OpenAI-compatible em /v1)
if llm_client is None:
    try:
        candidate = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
        _ = candidate.chat.completions.create(
            model=OLLAMA_LLM_MODEL,
            messages=[{"role": "user", "content": "ok"}],
            max_tokens=2, temperature=0.0,
        )
        llm_client, MODEL_NAME, LLM_PROVIDER = candidate, OLLAMA_LLM_MODEL, "ollama"
        print(f"✅ LLM via Ollama (fallback) | {OLLAMA_BASE_URL} | modelo: {MODEL_NAME}")
    except Exception as e:
        print(f"❌ Nenhum provedor LLM disponível: {e}")
        print("   Configure GROQ_API_KEY no .env ou rode `ollama serve` localmente.")


def llm_call(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
    """Chama o LLM ativo (Groq ou Ollama) e retorna o texto gerado."""
    if llm_client is None:
        return ""
    try:
        resp = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️  LLM ({LLM_PROVIDER}) falhou: {e}")
        return ""


if llm_client is not None:
    test = llm_call("Responda em uma palavra: qual é a capital do Brasil?")
    print(f"   Smoke test → '{test}' (esperado: Brasília)")

In [ ]:
from opensearchpy import OpenSearch

# Conectar ao OpenSearch
USE_OPENSEARCH = True  # se False, todo o pipeline usa fallback FAISS local

os_client = None
try:
    os_client = OpenSearch(
        hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
        http_auth=("admin", os.getenv("OPENSEARCH_PASSWORD", "admin")),
        use_ssl=False,
        verify_certs=False,
        timeout=10,
    )
    info = os_client.info()
    print(f"✅ OpenSearch conectado! Versão: {info['version']['number']}")
except Exception as e:
    print(f"⚠️  OpenSearch indisponível: {e}")
    print("   O lab cairá automaticamente em fallback FAISS local.")
    os_client = None
    USE_OPENSEARCH = False

## 📥 Passo 3 — Indexar o Corpus Jurídico

Antes de executar as técnicas de query rewriting, precisamos do corpus indexado para medir o
impacto no retrieval (Passo 7). Esta seção:

1. **Carrega o corpus** `corpus_juridico_aula3.json` (13 documentos: acórdãos STF/STJ + legislação CPP/CF)
2. **Gera embeddings BGE-M3** via Ollama (1024 dims, multilíngue) — fallback automático para HuggingFace `BAAI/bge-m3`
3. **Cria o índice kNN no OpenSearch** com mapping híbrido: campo textual em português (BM25) + campo vetorial (HNSW/cosseno)
4. **Indexa em bulk** — se OpenSearch estiver indisponível, monta um **vector store FAISS local** automaticamente

> ✅ Após este passo, o índice `corpus_juridico` (ou o FAISS em memória) está pronto para a busca híbrida do Passo 7.

In [ ]:
# ── Embeddings BGE-M3: Ollama (primário) + HuggingFace (fallback) ─

EMBED_DIM = 1024  # BGE-M3 dimension
embeddings = None
EMBED_BACKEND = None

# Tenta Ollama (consistente com servidor LLM local da Aula 1)
try:
    from langchain_ollama import OllamaEmbeddings
    cand = OllamaEmbeddings(
        model=OLLAMA_EMBED_MODEL,
        base_url=OLLAMA_BASE_URL,
    )
    _ = cand.embed_query("teste de conexão")
    embeddings = cand
    EMBED_BACKEND = f"Ollama ({OLLAMA_EMBED_MODEL})"
    print(f"✅ Embeddings via Ollama: {OLLAMA_EMBED_MODEL}")
except Exception as e:
    print(f"⚠️  Ollama embeddings indisponível ({e}); tentando HuggingFace…")

# Fallback HuggingFace (BAAI/bge-m3 — mesmo espaço vetorial)
if embeddings is None:
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        from langchain_community.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
    EMBED_BACKEND = "HuggingFace (BAAI/bge-m3)"
    print(f"✅ Embeddings via HuggingFace (fallback): BAAI/bge-m3")

print(f"   Backend ativo: {EMBED_BACKEND} | dim={EMBED_DIM}")

In [ ]:
# ── Carregar corpus jurídico (acórdãos + legislação) ─────────────

DATASET_PATH = "../datasets/corpus_juridico_aula3.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

documentos = dataset["documentos"]
print(f"✅ Corpus carregado: {len(documentos)} documentos")
print()
print("📋 Amostra:")
for d in documentos[:3]:
    fonte = d.get("numero", d["id"])
    ementa = d.get("ementa", d.get("texto", ""))[:80]
    print(f"   [{d['id']}] {fonte} — {ementa}…")

In [ ]:
# ── Criar índice OpenSearch com mapping híbrido (texto PT-BR + kNN) ─

INDEX_MAPPING = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 100,
        }
    },
    "mappings": {
        "properties": {
            "texto":          {"type": "text", "analyzer": "portuguese"},
            "ementa":         {"type": "text", "analyzer": "portuguese"},
            "palavras_chave": {"type": "keyword"},
            "tipo":           {"type": "keyword"},
            "tribunal":       {"type": "keyword"},
            "numero":         {"type": "keyword"},
            "data":           {"type": "date", "ignore_malformed": True},
            "embedding": {
                "type": "knn_vector",
                "dimension": EMBED_DIM,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "faiss",
                    "parameters": {"ef_construction": 128, "m": 16},
                },
            },
        }
    },
}

if USE_OPENSEARCH and os_client is not None:
    try:
        if os_client.indices.exists(index=INDEX_NAME):
            print(f"ℹ️  Índice '{INDEX_NAME}' já existe — recriando para garantir mapping atualizado")
            os_client.indices.delete(index=INDEX_NAME)
        os_client.indices.create(index=INDEX_NAME, body=INDEX_MAPPING)
        print(f"✅ Índice '{INDEX_NAME}' criado com mapping kNN (HNSW/cosseno, dim={EMBED_DIM})")
    except Exception as e:
        print(f"⚠️  Falha ao criar índice OpenSearch: {e}")
        print("   Caindo para fallback FAISS local…")
        USE_OPENSEARCH = False
else:
    print("ℹ️  OpenSearch desabilitado — usando fallback FAISS direto")

In [ ]:
# ── Indexar documentos: gera embeddings + faz bulk no OpenSearch ──
# ── Fallback automático: monta vector store FAISS local             ──

from opensearchpy.helpers import bulk

faiss_store = None  # será preenchido apenas se OpenSearch falhar

# Pré-computa todos os embeddings de uma vez (1 chamada/doc no Ollama)
print(f"⏳ Gerando embeddings para {len(documentos)} documentos com {EMBED_BACKEND}…")
t0 = time.time()
textos_para_embed = [
    f"{d.get('ementa', '')}\n\n{d.get('texto', '')}".strip()
    for d in documentos
]
vetores = embeddings.embed_documents(textos_para_embed)
print(f"✅ {len(vetores)} embeddings gerados em {time.time()-t0:.1f}s")

if USE_OPENSEARCH and os_client is not None:
    actions = []
    for doc, vec in zip(documentos, vetores):
        actions.append({
            "_index": INDEX_NAME,
            "_id":    doc["id"],
            "_source": {
                "texto":          doc.get("texto", ""),
                "ementa":         doc.get("ementa", ""),
                "palavras_chave": doc.get("palavras_chave", []),
                "tipo":           doc.get("tipo", ""),
                "tribunal":       doc.get("tribunal", ""),
                "numero":         doc.get("numero", doc["id"]),
                "data":           doc.get("data", ""),
                "embedding":      vec,
            },
        })
    try:
        success, errors = bulk(os_client, actions, refresh="wait_for")
        print(f"✅ Bulk OpenSearch: {success} docs indexados; erros={len(errors) if isinstance(errors, list) else errors}")
        count = os_client.count(index=INDEX_NAME)["count"]
        print(f"   Total no índice '{INDEX_NAME}': {count}")
    except Exception as e:
        print(f"⚠️  Bulk OpenSearch falhou: {e}")
        print("   Caindo para FAISS local…")
        USE_OPENSEARCH = False

if not USE_OPENSEARCH:
    # Fallback FAISS local — mesmo espaço vetorial, sem servidor
    from langchain_community.vectorstores import FAISS
    from langchain_core.documents import Document as LCDocument

    lc_docs = [
        LCDocument(
            page_content=textos_para_embed[i],
            metadata={
                "id":     d["id"],
                "fonte":  d.get("numero", d["id"]),
                "tipo":   d.get("tipo", ""),
                "ementa": d.get("ementa", ""),
            },
        )
        for i, d in enumerate(documentos)
    ]
    faiss_store = FAISS.from_documents(lc_docs, embeddings)
    print(f"✅ FAISS local construído: {len(lc_docs)} documentos | dim={EMBED_DIM}")

print()
print(f"📊 Backend ativo: {'OpenSearch (' + INDEX_NAME + ')' if USE_OPENSEARCH else 'FAISS local (in-memory)'}")

## 📋 Passo 4 — Carregar Queries do Baseline

In [ ]:
# Carregar dataset com queries
DATASET_PATH = "../datasets/corpus_juridico_aula3.json"

try:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    queries_baseline = dataset["queries_baseline"]
    print(f"✅ Dataset carregado: {len(queries_baseline)} queries")
except Exception:
    # Fallback: queries hardcoded para demonstração
    queries_baseline = [
        {"id": "q_001", "query_original": "Podem prender alguém sem mandado?"},
        {"id": "q_002", "query_original": "Policial pode ler mensagens do celular do suspeito?"},
        {"id": "q_003", "query_original": "O suspeito é obrigado a falar na delegacia?"},
        {"id": "q_004", "query_original": "O advogado pode ver o inquérito policial?"},
        {"id": "q_005", "query_original": "Como provar que o réu tinha intenção de matar?"},
    ]
    print("⚠️  Usando queries hardcoded (dataset não encontrado)")

print()
print("📋 Queries do baseline:")
for q in queries_baseline[:5]:
    print(f"  [{q['id']}] {q['query_original']}")

## 🔄 Passo 5 — Implementar as 3 Técnicas de Rewriting

In [ ]:
# ── TÉCNICA 1: Paraphrase Rewriting ──────────────────────────────

def paraphrase_rewriting(query: str, n: int = 3) -> List[str]:
    """Gera N reformulações técnicas da query jurídica."""
    prompt = f"""Você é especialista em direito processual penal brasileiro.
Reformule a query abaixo em {n} variações usando terminologia técnica jurídica.
Mantenha o mesmo significado, mas use linguagem técnica de documentos jurídicos.

Query original: {query}

Retorne APENAS as {n} variações, uma por linha, sem numeração ou introdução."""
    
    result = llm_call(prompt, temperature=0.5)
    if not result:
        return [query]  # fallback
    
    lines = [l.strip() for l in result.split('\n') if l.strip()]
    return lines[:n] if lines else [query]


# ── TÉCNICA 2: HyDE-Lite ─────────────────────────────────────────

def hyde_lite(query: str) -> str:
    """Gera parágrafo hipotético técnico para uso como query de busca."""
    prompt = f"""Você é redator jurídico especializado em processo penal brasileiro.
Escreva um parágrafo curto (4-6 linhas) que seria encontrado em acórdão ou código 
e que responderia diretamente à pergunta. Use linguagem técnica formal.

Pergunta: {query}

Escreva APENAS o parágrafo, sem título."""
    
    result = llm_call(prompt, temperature=0.2, max_tokens=300)
    return result if result else query


# ── TÉCNICA 3: Step-Back ─────────────────────────────────────────

def step_back(query: str) -> str:
    """Abstrai a query para nível conceitual mais geral."""
    prompt = f"""Você é especialista em direito processual penal.
Dada a pergunta específica abaixo, formule uma pergunta MAIS GERAL 
que abrange o conceito jurídico por trás dela.

Pergunta específica: {query}

Retorne APENAS a pergunta geral, sem explicação."""
    
    result = llm_call(prompt, temperature=0.1, max_tokens=100)
    return result if result else query

print("✅ Três técnicas de rewriting implementadas:")
print("   1. paraphrase_rewriting(query, n=3)")
print("   2. hyde_lite(query)")
print("   3. step_back(query)")

## 🧪 Passo 6 — Executar Rewriting nas 5 Queries

In [ ]:
# Processar todas as queries do baseline
print("⏳ Processando queries com as 3 técnicas de rewriting...")
print("   (Isso pode levar 1-2 minutos dependendo do LLM)")
print()

resultados_rewriting = []

for q in queries_baseline[:5]:
    qid = q["id"]
    original = q["query_original"]
    print(f"Processando {qid}: '{original}'")
    
    t0 = time.time()
    
    # Aplicar as 3 técnicas
    paraphrases = paraphrase_rewriting(original, n=2)
    hyde_doc    = hyde_lite(original)
    stepback_q  = step_back(original)
    
    elapsed = time.time() - t0
    
    resultados_rewriting.append({
        "id": qid,
        "original": original,
        "paraphrase_1": paraphrases[0] if len(paraphrases) > 0 else original,
        "paraphrase_2": paraphrases[1] if len(paraphrases) > 1 else original,
        "hyde_lite": hyde_doc[:200] + "..." if len(hyde_doc) > 200 else hyde_doc,
        "step_back": stepback_q,
        "tempo_s": round(elapsed, 2)
    })
    
    print(f"  ✅ Concluído em {elapsed:.1f}s")

print()
print(f"✅ {len(resultados_rewriting)} queries processadas!")

## 📊 Passo 7 — Tabela Comparativa

In [ ]:
# Exibir tabela comparativa
print("📊 TABELA COMPARATIVA: Query Original vs Reformulações")
print("=" * 80)
print()

for r in resultados_rewriting:
    print(f"━━━ [{r['id']}] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"ORIGINAL:     {r['original']}")
    print(f"PARAPHRASE 1: {r['paraphrase_1']}")
    print(f"PARAPHRASE 2: {r['paraphrase_2']}")
    print(f"HYDE-LITE:    {r['hyde_lite'][:120]}...")
    print(f"STEP-BACK:    {r['step_back']}")
    print(f"Tempo:        {r['tempo_s']}s")
    print()

# Salvar como CSV para análise
df = pd.DataFrame(resultados_rewriting)
df.to_csv("/tmp/rewriting_comparativo.csv", index=False, encoding="utf-8")
print("✅ Resultados salvos em /tmp/rewriting_comparativo.csv")

## 🔎 Passo 8 — Testar Impacto no Retrieval

In [ ]:
def buscar_corpus(query: str, top_k: int = 5) -> List[Dict]:
    """Busca híbrida no corpus indexado.

    OpenSearch: combina BM25 (texto + ementa) com kNN (embedding BGE-M3)
                usando script_score para fusão linear (0.4*texto + 0.6*vetor).
    FAISS:      busca por similaridade pura (embedding bge-m3).
    """
    # ── Backend OpenSearch (híbrido BM25 + kNN) ─────────────────────
    if USE_OPENSEARCH and os_client is not None:
        try:
            q_vec = embeddings.embed_query(query)
            body = {
                "size": top_k,
                "query": {
                    "script_score": {
                        "query": {
                            "multi_match": {
                                "query": query,
                                "fields": ["texto^1.5", "ementa^2", "palavras_chave"],
                                "type":  "best_fields",
                            }
                        },
                        "script": {
                            "source": "_score * 0.4 + cosineSimilarity(params.q, doc['embedding']) * 0.6 + 1.0",
                            "params": {"q": q_vec},
                        },
                    }
                },
            }
            resp = os_client.search(index=INDEX_NAME, body=body)
            hits = resp["hits"]["hits"]
            return [
                {
                    "id":    h["_id"],
                    "score": round(h["_score"], 3),
                    "texto": (h["_source"].get("texto", "")[:200] + "…"),
                    "fonte": h["_source"].get("numero", h["_id"]),
                }
                for h in hits
            ]
        except Exception as e:
            print(f"⚠️  Hybrid search falhou ({e}); tentando FAISS…")

    # ── Fallback FAISS (vetorial puro) ──────────────────────────────
    if faiss_store is not None:
        results = faiss_store.similarity_search_with_score(query, k=top_k)
        return [
            {
                "id":    d.metadata.get("id", ""),
                "score": round(float(s), 3),
                "texto": d.page_content[:200] + "…",
                "fonte": d.metadata.get("fonte", d.metadata.get("id", "")),
            }
            for d, s in results
        ]

    # ── Último recurso: retorno simulado (mantém didática) ──────────
    return [
        {"id": "doc_sim_1", "score": 0.95, "texto": "Resultado simulado…", "fonte": "CF/88 Art. 5º"},
        {"id": "doc_sim_2", "score": 0.88, "texto": "Resultado simulado…", "fonte": "CPP Art. 302"},
    ]


# Alias para retrocompatibilidade com nome anterior
buscar_opensearch = buscar_corpus

# Testar com a primeira query
query_teste = resultados_rewriting[0]
print(f"🔎 Testando retrieval para: [{query_teste['id']}]")
print()

print("QUERY ORIGINAL:")
res_original = buscar_corpus(query_teste["original"])
for r in res_original:
    print(f"  [{r['score']:.3f}] {r['fonte']}")

print()
print("QUERY STEP-BACK (mais abrangente):")
res_stepback = buscar_corpus(query_teste["step_back"])
for r in res_stepback:
    print(f"  [{r['score']:.3f}] {r['fonte']}")

# Calcular overlap
ids_original = {r["id"] for r in res_original}
ids_stepback = {r["id"] for r in res_stepback}
overlap = len(ids_original & ids_stepback) / max(len(ids_original), 1)
novos = len(ids_stepback - ids_original)

print()
print(f"📊 Análise de diversidade:")
print(f"   Backend: {'OpenSearch (hybrid)' if USE_OPENSEARCH else 'FAISS local'}")
print(f"   Overlap entre técnicas: {overlap:.0%}")
print(f"   Documentos novos recuperados pelo step-back: {novos}")

## ✅ Passo 9 — Reflexão e Análise

In [ ]:
print("📝 ANÁLISE FINAL — Query Rewriting")
print("=" * 60)
print()
print("Perguntas para reflexão:")
print()
print("1️⃣  Qual técnica gerou queries mais diferentes da original?")
print("    → Compare lexicalmente: HyDE-Lite tende a ser mais longa e técnica")
print()
print("2️⃣  Para queries jurídicas coloquiais, qual técnica é mais eficaz?")
print("    → Step-back é eficaz quando há forte jargão operacional vs formal")
print()
print("3️⃣  O HyDE-Lite pode causar problemas. Quais?")
print("    → RISCO: Se o LLM alucinar no parágrafo hipotético,")
print("       a busca vai recuperar documentos irrelevantes.")
print("       Em ambiente jurídico, isso é especialmente crítico!")
print()
print("4️⃣  Qual o custo de cada técnica?")
print("    Paraphrase: ~100 tokens | HyDE-Lite: ~200 tokens | Step-back: ~50 tokens")
print()
print("✅ LAB 1 CONCLUÍDO!")
print()
print("📋 CHECKLIST DE ENTREGA:")
print("   [ ] 3 técnicas implementadas e executadas")
print("   [ ] Tabela comparativa gerada para 5 queries")
print("   [ ] Teste de retrieval com query original vs reformulada")
print("   [ ] Arquivo CSV salvo em /tmp/rewriting_comparativo.csv")
print("   [ ] Reflexão respondida nos comentários abaixo")

## 📝 Espaço para Reflexão

*Complete aqui suas observações:*

**Qual técnica apresentou maior diversidade semântica?**
→ *[sua resposta]*

**Para o domínio jurídico policial (jargão operacional), qual técnica recomendaria?**
→ *[sua resposta]*

**Algum caso onde o HyDE-Lite gerou resultado inadequado?**
→ *[sua resposta]*
